# Steering-vector extraction — atomic cognitive mechanisms

Phase 1 (**vLLM**, HF fallback): generate persona-conditioned completions over a neutral prompt bank.
Phase 2 (**repeng**, HF transformers): build ON/baseline contrastive pairs and train one control
vector per mechanism (PCA on last-token hidden-state differences).

This run produces **two bundles** on the *same* neutral prompts:

* `control_vectors.pkl` — the 10 **clinical** mechanisms (`personas.py`): rumination, worry,
  threat-vigilance, … . Used by the steering notebook's `looping` arm.
* `control_vectors_pc.pkl` — the **predictive-coding primitives** (`personas_pc.py`):
  `prior_precision_high`, `circular_inference`, … . The steering notebook's **REBUS** arm uses
  `prior_precision_high`; without this bundle REBUS falls back to a coarse −coeff on the looping
  vector.

**Model:** default `google/gemma-4-12B-it` (full bf16; override via `STEER_MODEL` env or
`config.MODEL_NAME`). Gemma 4 is an encoder-free unified multimodal model — we run it **text-only**
and **with thinking suppressed** (`enable_thinking=False`, handled in the chat-template helpers) so
the completions are clean training data.

**Generations are saved to Google Drive**, so if the runtime breaks you can restart, re-run the
setup cells, and jump straight to Phase 2 — it reloads the dataset from Drive. The vLLM engine is
explicitly ejected before Phase 2 re-loads the model in HF transformers (vLLM can't expose hidden
states).

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Clone the project so `from steering import ...` works on a fresh Colab runtime.
REPO_URL = "https://github.com/ChuloIva/Predictive_coding"

def _find_lab():
    here = Path.cwd()
    for cand in (here, here / "steering_lab"):          # already inside the repo / lab?
        if (cand / "steering" / "extract.py").exists():
            return cand
    dest = here / "Predictive_coding"                    # else clone it
    if not dest.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    return dest / "steering_lab"

LAB_DIR = _find_lab()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))                         # so `from steering import ...` resolves
print("cwd:", os.getcwd())

In [ ]:
# Deps. Phase 2 (extraction) needs only repeng + a recent transformers. Phase 1 (generation) uses
# vLLM for speed — but Gemma 4 (encoder-free multimodal) is supported only on a NIGHTLY vLLM wheel
# (support landed post-stable, vllm PR #44429). If the nightly install fails or its CUDA tag doesn't
# match this runtime, Phase 1 auto-falls back to HF transformers (slower, but always works).
%pip install -q -U transformers accelerate
%pip install -q repeng sentencepiece

# Nightly vLLM (CUDA 12.9 wheels). Change cu129 to match the runtime's CUDA, or comment this out to
# force the HF fallback. The `|| echo` keeps the notebook running if the wheel won't install.
!pip install -q -U --pre vllm --extra-index-url https://wheels.vllm.ai/nightly/cu129 --extra-index-url https://download.pytorch.org/whl/cu129 || echo "nightly vLLM install failed -> Phase 1 will use the HF fallback"

In [ ]:
# API keys — only HF_TOKEN matters here, and only for gated models (Gemma 4 is Apache-2.0/ungated).
import os
from pathlib import Path
_KEYS = ["HF_TOKEN"]
try:
    from google.colab import userdata  # type: ignore
    for k in _KEYS:
        if not os.environ.get(k):
            try:
                v = userdata.get(k)
                if v: os.environ[k] = v
            except Exception: pass
except ImportError:
    pass
try:
    from dotenv import load_dotenv
    for p in [LAB_DIR / ".env", Path.cwd() / ".env"]:
        if p.exists(): load_dotenv(p, override=False)
except ImportError:
    pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for ungated models)")

### Mount Drive & define persistent paths
Generations + trained vectors are written to Drive so they survive a runtime restart. Two persona
sets → two generation files → two bundles.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/steering_pilot")
except Exception as e:
    print("Drive not mounted (", e, ") — falling back to local dir")
    OUT_DIR = LAB_DIR / "steering" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# clinical bundle (looping arm) + PC-primitive bundle (REBUS arm)
GEN_PATH        = str(OUT_DIR / "generations.jsonl")
GEN_PC_PATH     = str(OUT_DIR / "generations_pc.jsonl")
VECTORS_PATH    = str(OUT_DIR / "control_vectors.pkl")
VECTORS_PC_PATH = str(OUT_DIR / "control_vectors_pc.pkl")
print("artifacts ->", OUT_DIR)
print("clinical generations exist?", Path(GEN_PATH).exists(),
      "| PC generations exist?", Path(GEN_PC_PATH).exists())

## 2. Phase 1 — generate data

For each persona set: every mechanism × {on, baseline} × 50 neutral prompts → one completion,
written to its `generations*.jsonl` on Drive. **Skip a set** automatically if its file already
exists (so a restart jumps straight to Phase 2).

Engine: vLLM if it can load the model (text-only, thinking off), else an automatic HF-transformers
fallback via the same robust loader the steering notebook uses.

In [ ]:
from steering import config, generate, steer
from steering.personas import PERSONAS, NEUTRAL_PROMPTS
from steering.personas_pc import PC_PERSONAS

MODEL_NAME = os.environ.get("STEER_MODEL", config.MODEL_NAME)   # default google/gemma-4-12B-it
gen_cfg = config.GenConfig(model_name=MODEL_NAME)
# gen_cfg.n_samples = 2; gen_cfg.max_tokens = 256   # tweak for more / longer data (vLLM path only)

# clinical + PC primitives, on the SAME neutral prompts.
SETS = [("clinical", PERSONAS, GEN_PATH), ("pc", PC_PERSONAS, GEN_PC_PATH)]

# A real tokenizer with the chat template (Gemma 4 exposes it via the processor, not AutoTokenizer).
tokenizer = steer._load_tokenizer(MODEL_NAME, os.environ.get("HF_TOKEN"))

# Try vLLM once; fall back to HF if it can't load this model. Reuse the engine across both sets.
llm = hf_model = None
USE_VLLM = True
try:
    from vllm import LLM
    llm = LLM(model=MODEL_NAME, dtype=gen_cfg.dtype, max_model_len=gen_cfg.max_model_len,
              gpu_memory_utilization=gen_cfg.gpu_memory_utilization, trust_remote_code=True,
              limit_mm_per_prompt={"image": 0, "audio": 0})   # text-only: skip vision/audio embedders
    print("Phase 1 engine: vLLM")
except Exception as e:
    USE_VLLM = False
    print("vLLM unavailable for", MODEL_NAME, "(", str(e)[:160], ") -> HF fallback")
    hf_model, tokenizer = steer.load_model_and_tokenizer(
        MODEL_NAME, dtype=gen_cfg.dtype, device_map="auto", hf_token=os.environ.get("HF_TOKEN"))

records_by_set = {}
for name, personas, gpath in SETS:
    if Path(gpath).exists():
        print(f"[{name}] generations already on disk -> skip ({gpath})")
        continue
    print(f"[{name}] {len(personas)} mechanisms x 2 variants x {len(NEUTRAL_PROMPTS)} prompts ...")
    if USE_VLLM:
        recs = generate.generate_dataset(
            gen_cfg, personas=personas, out_path=gpath, llm=llm, tokenizer=tokenizer)
    else:
        recs = generate.generate_dataset_hf(
            gen_cfg, personas=personas, out_path=gpath, model=hf_model, tokenizer=tokenizer)
    records_by_set[name] = recs
    print(f"[{name}] {len(recs)} records -> {gpath}")

In [ ]:
# Tier-0 eyeball: does the ON persona actually exhibit the mechanism vs its matched baseline?
def peek(name, mech, prompt_idx=0, n=500):
    recs = records_by_set.get(name, [])
    for variant in ("on", "baseline"):
        r = next((x for x in recs if x["mechanism"] == mech
                  and x["variant"] == variant and x["prompt_idx"] == prompt_idx), None)
        if r:
            print(f"\n[{name}/{mech}/{variant}]  prompt: {r['prompt']}\n{r['completion'][:n]}")

if records_by_set:
    peek("clinical", "rumination")
    peek("pc", "prior_precision_high")
else:
    print("generations were loaded from Drive (not regenerated) — skip the eyeball, or reload a file")

### Eject the Phase-1 engine
Free all GPU memory held by vLLM (or the HF fallback model) before Phase 2 re-loads the model for
hidden-state extraction.

In [ ]:
import gc, torch
for _name in ("llm", "hf_model"):
    if _name in globals() and globals()[_name] is not None:
        try: del globals()[_name]
        except Exception: pass
try:
    from vllm.distributed.parallel_state import (destroy_model_parallel,
                                                  destroy_distributed_environment)
    destroy_model_parallel(); destroy_distributed_environment()
except Exception as e:
    print("vllm teardown skipped:", e)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU allocated:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
# If memory isn't fully freed (vLLM can be sticky): Runtime -> Restart, re-run cells 1-6, then skip
# Phase 1 and run Phase 2 — it reloads generations from Drive.

## 3. Phase 2 — extract control vectors (repeng)

Loads the model fresh in HF transformers (needed for hidden states) via the **robust loader**
(Gemma-4/multimodal aware — sets `model.repeng_layers` so repeng finds the text decoder), then
trains one `ControlVector` per mechanism for **both** persona sets.

Self-contained: after a runtime restart, run cells 1–6 then come straight here.

In [ ]:
import os, torch
from steering import steer, config

MODEL_NAME = os.environ.get("STEER_MODEL", config.MODEL_NAME)
ecfg = config.ExtractConfig(model_name=MODEL_NAME)
# Optional: restrict to a middle band where style/persona usually steers cleanest:
# ecfg.hidden_layers = list(range(-6, -19, -1))

# Robust loader: handles Gemma 4's multimodal wrapper, unwraps the processor tokenizer, and sets
# model.repeng_layers so repeng's ControlVector.train finds the text decoder (config.num_hidden_layers
# would be wrong for a multimodal wrapper — use len(decoder_layers(model)) instead).
model, tok = steer.load_model_and_tokenizer(
    MODEL_NAME, dtype=ecfg.dtype, device_map="cuda", hf_token=os.environ.get("HF_TOKEN"))
model.eval()
print("decoder layers:", len(steer.decoder_layers(model)), "| model:", MODEL_NAME)

In [ ]:
from steering import extract
from steering.personas import PERSONA_BY_ID
from steering.personas_pc import PC_PERSONA_BY_ID

# (set name, generations path, output bundle, persona registry to read ON/baseline prompts from)
JOBS = [
    ("clinical", GEN_PATH,    VECTORS_PATH,    PERSONA_BY_ID),
    ("pc",       GEN_PC_PATH, VECTORS_PC_PATH, PC_PERSONA_BY_ID),
]

vectors_by_set = {}
for name, gpath, vpath, pbid in JOBS:
    if not Path(gpath).exists():
        print(f"[{name}] no generations at {gpath} — skip (run its Phase 1, or check the path)")
        continue
    records = extract.load_records(gpath)
    pairs = extract.build_pairs(records, tok, ecfg, persona_by_id=pbid)
    print(f"[{name}] loaded {len(records)} records | pairs:", {m: len(p) for m, p in pairs.items()})
    vectors = extract.extract_vectors(model, tok, pairs, ecfg)
    meta = str(Path(vpath).with_name(Path(vpath).stem + "_meta.json"))
    extract.save_bundle(vectors, vpath, model_name=ecfg.model_name, cfg=ecfg,
                        pairs=pairs, meta_path=meta)
    vectors_by_set[name] = vectors
    print(f"[{name}] trained {len(vectors)} vectors -> {vpath}\n")

In [ ]:
# Separability check per set: cosine-similarity matrix between mechanism directions at a mid layer.
from steering import extract

def show_cosine(name):
    vectors = vectors_by_set.get(name)
    if not vectors:
        return
    any_v = next(iter(vectors.values()))
    layers = sorted(any_v.directions.keys())
    layer = layers[len(layers) // 2]
    labels, mat = extract.cosine_matrix(vectors, layer)
    print(f"\n[{name}] cosine similarity @ layer {layer}\n")
    print("              " + "".join(f"{l[:8]:>9s}" for l in labels))
    for i, l in enumerate(labels):
        print(f"{l[:13]:13s} " + "".join(f"{mat[i,j]:+9.2f}" for j in range(len(labels))))

show_cosine("clinical")
show_cosine("pc")

In [ ]:
# Bundles are already saved to Drive (cell above). Optionally download them locally too.
try:
    from google.colab import files  # type: ignore
    for vpath in (VECTORS_PATH, VECTORS_PC_PATH):
        if Path(vpath).exists():
            files.download(vpath)
            files.download(str(Path(vpath).with_name(Path(vpath).stem + "_meta.json")))
except ImportError:
    print("Not on Colab — bundles at:", VECTORS_PATH, "and", VECTORS_PC_PATH)

## Next: steering notebook

Open `steer_mechanisms.ipynb`. Point its bundle loader at `control_vectors.pkl` (clinical) — and
`control_vectors_pc.pkl` is auto-detected for the REBUS arm. Both must be extracted on the **same
model** you steer (vectors don't transfer across models).

```python
from steering import extract
bundle = extract.load_bundle("control_vectors.pkl")
vecs = extract.bundle_to_control_vectors(bundle)   # {mechanism: ControlVector}
# Experiment ③ looping arm: vecs['rumination']; REBUS arm: PC vecs['prior_precision_high'].
# Compose syndromes: vecs['rumination'] + vecs['negative_self_schema'] + vecs['hopelessness']
```
Composition recipes are in `personas.SYNDROME_RECIPES` / `personas.COMPOUND_PREDICTIONS`.